# Pseudotime analysis

https://github.com/theislab/cellrank/issues/592

In [ ]:
import cellrank as cr

import scanpy as sc
import scvelo as scv

import pickle

from scipy import sparse
import pandas as pd
import numpy as np
import math

import matplotlib.pyplot as plt

import os

## Settings

In [ ]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning) 

# Warnings scanpy off
sc.settings.verbosity = 0

## Plotting 

In [ ]:
# Scanpy figures
cm_to_inch = lambda x: x / 2.54
sc.set_figure_params(figsize=(cm_to_inch(5), cm_to_inch(5)), dpi_save=300, fontsize=6, frameon=False, color_map='Reds', facecolor='white')
scv.set_figure_params(figsize=(cm_to_inch(5), cm_to_inch(5)), dpi_save=300, fontsize=6, frameon=False, color_map='Reds', facecolor='white')

# Load external scores 

In [ ]:
ms = pd.read_csv('result/ms/ms.csv', index_col=0)
grn = pd.read_csv("result/scenic/res_0/auc_mtx.csv", index_col=0)

# Helper functions

In [ ]:
# Combine kernels
def kernel_workflow(k_1, k_2, kr_1=1.0, kr_2=1.0): 

    # Set count matrix
    k_1.adata = k_1.adata.raw.to_adata()
    k_2.adata = k_2.adata.raw.to_adata()

    # Set raw 
    k_1.adata.raw = k_1.adata
    k_2.adata.raw = k_2.adata
    
    # Transission matrix
    k_1 = k_1.compute_transition_matrix()
    k_2 = k_2.compute_transition_matrix()
    
    k = kr_1*k_1 + kr_2*k_2
    
    return(k)

In [ ]:
# GPCCA
def gpcca_workflow(k, n_components): 
    
    # Coarse-graining transition matrix into macro-states with GPCCA
    g = cr.estimators.GPCCA(k)
    g.compute_schur(n_components=n_components)
    
    return(g)

# CellRank 

# CD4+ T-cells

In [ ]:
adata_0 = sc.read_h5ad('data/object/scvi/pseudotime/adata_0.h5ad')
adata_0.obs = adata_0.obs.merge(ms, left_index=True, right_index=True, how='left')
adata_0.obs = adata_0.obs.merge(grn, left_index=True, right_index=True, how='left')

# Leiden 
sc.tl.leiden(adata_0, resolution=1.00, flavor='igraph', n_iterations=2, key_added='leiden_paga')
        
# PAGA 
sc.tl.paga(adata_0, groups='leiden_paga', model='v1.2')
sc.pl.paga(adata_0, layout='fr', threshold=0, plot=False)

# Diffusion pseudotime 
adata_0.uns['iroot'] = np.flatnonzero((adata_0.obs['celltype_low']=='$T_{h}\ N/CM$'))[0]
sc.tl.diffmap(adata_0, n_comps=15)
sc.tl.dpt(adata_0, n_dcs=10)

# Normalize for plotting 
sc.pp.normalize_total(adata_0)
sc.pp.log1p(adata_0)

In [ ]:
scores = ['leiden_paga', 'TRM1_vs_TRM17_ucell', 'TRM17_vs_TRM1_ucell', 'IL7R', 'CCR6', 'CCR7', 'RORC', 'RORC(+)', 'MAF', 'MAF(+)', 'STAT3', 'STAT3(+)', 'ZNF683', 'ZNF683(+)', 'TBX21', 'TBX21(+)']
nplots = len(scores)
ncols = 8
nrows = math.ceil(nplots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(cm_to_inch(6) * ncols, cm_to_inch(5) * nrows))
axes = axes.flatten()

for i, score in enumerate(scores):
    sc.pl.umap(adata_0, color=score, add_outline=True, size=20, legend_loc="none", show=False, use_raw=False, ax=axes[i])
    if score=='leiden_paga': 
        sc.pl.umap(adata_0, color='leiden_paga', add_outline=True, size=20, legend_loc="none", show=False, ax=axes[i])
    if score!='leiden_paga':
        sc.pl.umap(adata_0, color='celltype_low', add_outline=True, size=20, legend_loc="none", show=False, ax=axes[i])
    scv.pl.paga(adata_0, basis="umap", threshold=0.10, fontsize=6, node_size_scale=0.5, color=score, legend_loc='on data', show=False, title=score, use_raw=False, ax=axes[i])

# Hide unused subplots if any
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('result/paga/paga_cd4pos_celltype_low.pdf', dpi=300)

In [ ]:
adata_0.obs['t_cell_subset'] = adata_0.obs.apply(lambda row: 'T17hi' if row['TRM17_vs_TRM1_ucell'] >= 0.35 else 'T17lo', axis=1)

scores = ['t_cell_subset', 'TRM1_vs_TRM17_ucell', 'TRM17_vs_TRM1_ucell', 'IL7R', 'CCR6', 'CCR7', 'RORC', 'RORC(+)', 'MAF', 'MAF(+)', 'STAT3', 'STAT3(+)', 'ZNF683', 'ZNF683(+)', 'TBX21', 'TBX21(+)']
nplots = len(scores)
ncols = 8
nrows = math.ceil(nplots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(cm_to_inch(6) * ncols, cm_to_inch(5) * nrows))
axes = axes.flatten()

for i, score in enumerate(scores):
    sc.pl.umap(adata_0, color=score, add_outline=True, size=20, legend_loc="none", show=False, use_raw=False, ax=axes[i])
    if score != 't_cell_subset':
        scv.pl.paga(adata_0, basis="umap", threshold=0.10, fontsize=6, node_size_scale=0.5, color=score, legend_loc='on data', show=False, title=score, use_raw=False, ax=axes[i])

# Hide unused subplots if any
for j in range(i + 1, len(axes)):
    axes[j].axis('off')


# Hide unused subplots if any
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('result/paga/paga_cd4pos_score.pdf', dpi=300)

In [ ]:
compute = True

In [ ]:
if compute: 
        
    # Pseudotime kernel 
    from cellrank.kernels import PseudotimeKernel
    pk_0 = PseudotimeKernel(adata_0, time_key='dpt_pseudotime', backward=False)
        
    # Connectivity kernel 
    from cellrank.kernels import ConnectivityKernel
    ck_0 = ConnectivityKernel(adata_0)

    # Combine kernels
    k_0 = kernel_workflow(pk_0, ck_0, kr_1=0.5, kr_2=0.5)

    # Save kernel 
    k_0.write('data/object/cellrank/k_0.pickle', write_adata=True)

    # GPCCA
    g_0 = gpcca_workflow(k_0, n_components=10)

    # Fit macrostates 
    g_0.fit(n_states=5, n_cells=100, cluster_key="leiden_paga")
        
    g_0.write('data/object/cellrank/g_0.pickle', write_adata=True)
        
else: 

    k_0 = cr.kernels.PrecomputedKernel.read('data/object/cellrank/k_0.pickle')
    g_0 = cr.estimators.GPCCA.read('data/object/cellrank/g_0.pickle')

# CD8+ T-cells

In [ ]:
adata_1 = sc.read_h5ad('data/object/scvi/pseudotime/adata_1.h5ad')
adata_1.obs = adata_1.obs.merge(ms, left_index=True, right_index=True, how='left')
adata_1.obs = adata_1.obs.merge(grn, left_index=True, right_index=True, how='left')

# Leiden 
sc.tl.leiden(adata_1, resolution=1.0, flavor='igraph', n_iterations=2, key_added='leiden_paga')
sc.tl.leiden(adata_1, resolution=0.50, restrict_to=('leiden_paga', ['8']), flavor='igraph', n_iterations=2, key_added='leiden_paga')

adata_1.obs['leiden_paga'] = adata_1.obs['leiden_paga'].replace({'8,0': '8,1'})

# Set Leiden by cluster size 
cat_var = "leiden_paga"
counts = adata_1.obs[cat_var].value_counts()
ranks = counts.rank(ascending=False, method='dense').astype(int) - 1
adata_1.obs['leiden_paga'] = adata_1.obs[cat_var].map(ranks).astype(str)
        
# PAGA 
sc.tl.paga(adata_1, groups='leiden_paga', model='v1.2')
sc.pl.paga(adata_1, layout='fr', threshold=0, plot=False)

# Diffusion pseudotime 
adata_1.uns['iroot'] = np.flatnonzero((adata_1.obs['celltype_low']=='$T_{c}\ N/CM$') & (adata_1.obs['time_point']=='Baseline'))[0]
sc.tl.diffmap(adata_1, n_comps=15)
sc.tl.dpt(adata_1, n_dcs=15)

# Normalize for plotting 
sc.pp.normalize_total(adata_1)
sc.pp.log1p(adata_1)

In [ ]:
scores = ['leiden_paga', 'TRM1_vs_TRM17_ucell', 'TRM17_vs_TRM1_ucell', 'IL7R', 'CCR6', 'CCR7', 'RORC', 'RORC(+)', 'MAF', 'MAF(+)', 'STAT3', 'STAT3(+)', 'ZNF683', 'ZNF683(+)', 'TBX21', 'TBX21(+)']
nplots = len(scores)
ncols = 8
nrows = math.ceil(nplots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(cm_to_inch(6) * ncols, cm_to_inch(5) * nrows))
axes = axes.flatten()

for i, score in enumerate(scores):
    sc.pl.umap(adata_1, color=score, add_outline=True, size=20, legend_loc="none", show=False, use_raw=False, ax=axes[i])
    if score=='leiden_paga': 
        sc.pl.umap(adata_1, color='leiden_paga', add_outline=True, size=20, legend_loc="none", show=False, ax=axes[i])
    if score!='leiden_paga':
        sc.pl.umap(adata_1, color='celltype_low', add_outline=True, size=20, legend_loc="none", show=False, ax=axes[i])
    scv.pl.paga(adata_1, basis="umap", threshold=0.10, fontsize=6, node_size_scale=0.5, color=score, legend_loc='on data', show=False, title=score, use_raw=False, ax=axes[i])

# Hide unused subplots if any
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('result/paga/paga_cd8pos_celltype_low.pdf', dpi=300)

In [ ]:
adata_1.obs['t_cell_subset'] = adata_1.obs.apply(lambda row: 'T17hi' if row['TRM17_vs_TRM1_ucell'] >= 0.35 else 'T17lo', axis=1)


scores = ['t_cell_subset', 'TRM1_vs_TRM17_ucell', 'TRM17_vs_TRM1_ucell', 'IL7R', 'CCR6', 'CCR7', 'RORC', 'RORC(+)', 'MAF', 'MAF(+)', 'STAT3', 'STAT3(+)', 'ZNF683', 'ZNF683(+)', 'TBX21', 'TBX21(+)']
nplots = len(scores)
ncols = 8
nrows = math.ceil(nplots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(cm_to_inch(6) * ncols, cm_to_inch(5) * nrows))
axes = axes.flatten()

for i, score in enumerate(scores):
    sc.pl.umap(adata_1, color=score, add_outline=True, size=20, legend_loc="none", show=False, use_raw=False, ax=axes[i])
    if score != 't_cell_subset': 
        scv.pl.paga(adata_1, basis="umap", threshold=0.10, fontsize=6, node_size_scale=0.5, color=score, legend_loc='on data', show=False, title=score, use_raw=False, ax=axes[i])

# Hide unused subplots if any
for j in range(i + 1, len(axes)):
    axes[j].axis('off')


# Hide unused subplots if any
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('result/paga/paga_cd8pos_score.pdf', dpi=300)

In [ ]:
compute = True

In [ ]:
if compute: 
        
    # Pseudotime kernel 
    from cellrank.kernels import PseudotimeKernel
    pk_1 = PseudotimeKernel(adata_1, time_key='dpt_pseudotime')
        
    # Connectivity kernel 
    from cellrank.kernels import ConnectivityKernel
    ck_1 = ConnectivityKernel(adata_1)

    # Combine kernels
    k_1 = kernel_workflow(pk_1, ck_1, kr_1=0.50, kr_2=0.50)

    # Save kernel 
    k_1.write('data/object/cellrank/k_1.pickle', write_adata=True)

    # GPCCA
    g_1 = gpcca_workflow(k_1, n_components=10)

    # Fit macrostates 
    g_1.fit(n_states=10, n_cells=100, cluster_key="leiden_paga")
        
    g_1.write('data/object/cellrank/g_1.pickle', write_adata=True)
        
else: 

    k_1 = cr.kernels.PrecomputedKernel.read('data/object/cellrank/k_1.pickle')
    g_1 = cr.estimators.GPCCA.read('data/object/cellrank/g_1.pickle')